# 05 Développement de la solution

Ce notebook présente le modèle retenu pour prédire le prix de vente (`SalePrice`) des maisons d'Ames : les paramètres utilisés, les hypothèses retenues et les justifications techniques de ce choix. Il part des données préparées à l'[étape 4](04_pretraitement.ipynb). Le code est dans `src/training.py`.

**Modèle retenu : la moyenne simple de cinq modèles.** Lasso, ElasticNet et Kernel Ridge sont des modèles linéaires; Gradient Boosting et XGBoost sont des modèles d'arbres de décision. Chaque modèle prédit le logarithme du prix; nous faisons la moyenne des cinq prédictions, puis revenons en dollars.

**Évaluation.** Comme à l'étape 4, nous mesurons la RMSE des logarithmes des prix, la métrique de Kaggle, par validation croisée à cinq plis sur les 1 460 ventes de `train.csv` (`random_state=42`). Le prétraitement est réappris dans chaque pli, et les deux ventes aberrantes (524 et 1299) sont retirées de l'entraînement seulement.

**Plan.** La section 1 présente les approches essayées avant d'arriver à un ensemble de modèles, et la façon dont nous l'évaluons. Les sections 2 à 4 justifient le choix : pourquoi combiner des modèles, ce que chacun des cinq modèles apporte, et quels modèles garder d'après les données. La section 5 présente le modèle, ses paramètres et ses hypothèses; la section 6, ses résultats et la soumission Kaggle.

## 1. Les approches essayées

Avant d'arriver au modèle retenu, nous avons essayé plusieurs approches dans des expériences (`notebooks/experiences/`). Elles ont été comparées sur une même partition (80 % des ventes pour l'entraînement, 20 % pour la validation, `random_state=42`) et avec la même métrique : la RMSE des logarithmes des prix, celle de Kaggle. Les scores des sections 1.1 à 1.6 viennent de ces expériences. À partir de la section 1.7, nous n'utilisons plus de partition : le modèle est évalué par validation croisée sur toutes les ventes.

### 1.1 RAG : estimer à partir de ventes comparables

Dans un autre projet portant sur des maisons à vendre sur Centris, un RAG (*Retrieval-Augmented Generation*) donnait de bonnes estimations. Nous avons donc commencé par cette approche ([expérience 01](experiences/01_rag_filtre.ipynb)). Elle imite un évaluateur immobilier en trois temps :

1. **Indexation.** Chaque maison d'entraînement est décrite par un court texte. Ce texte est converti en vecteur (*embedding*) et stocké dans une base vectorielle, avec le prix de la maison.
2. **Recherche.** Pour une maison à estimer, nous retrouvons les cinq ventes dont la description est la plus proche, de préférence dans le même quartier.
3. **Génération.** Un grand modèle de langage (LLM) reçoit ces cinq comparables, leur prix et la description de la maison, puis estime son prix.

Cette première version obtient une RMSE log de 0,140.

### 1.2 RAG, version 2 : de meilleurs comparables

L'analyse des erreurs de la version 1 a montré trois faiblesses : des informations décisives manquaient dans le texte (les deux plus grandes erreurs étaient des maisons en zonage commercial, sans air climatisé central), la recherche abandonnait trop vite le quartier, et le premier critère de recherche, le type de logement (`MSSubClass`), était moins lié au prix que la qualité générale (`OverallQual`). La [version 2](experiences/02_rag_filtre_v2.ipynb) garde le même LLM et le même prompt, mais change la recherche et la description des maisons :

- **Recherche.** Les comparables doivent d'abord avoir une qualité générale à ±1 et une surface habitable à ±25 %, dans le même quartier tant que possible. 90 % des maisons trouvent cinq comparables à ce premier niveau.
- **Description.** Le texte passe de 19 à 33 champs (zonage, air climatisé, sous-sol, variables calculées comme l'âge et la surface totale), avec des valeurs lisibles plutôt que des codes.
- **Prix des comparables.** Le prompt donne aussi le prix au pied carré de chaque comparable.

Les comparables sont nettement meilleurs : de 0,140 à **0,133**. Il corrigeait déjà une bonne partie des défauts de la recherche en version 1.

### 1.3 XGBoost : un modèle conçu pour les données tabulaires

Les données d'Ames sont tabulaires : 79 colonnes de surfaces, de qualités, de codes et d'années. Les modèles de *gradient boosting* sur arbres de décision, comme XGBoost, fonctionnent généralement bien sur ce type de données. Ils captent les seuils et les interactions sans hypothèse de forme. Après réglage des hyperparamètres ([expérience 03](experiences/03_xgboost.ipynb)), XGBoost obtient une RMSE log de 0,125 à 0,130 selon la configuration : il fait mieux que le RAG.

**Pourquoi le RAG fait-il moins bien ici?** Dans l'autre projet, le RAG exploitait surtout des données textuelles qualitatives, comme la description rédigée de chaque maison. Le jeu de données de la compétition n'en contient pas : notre texte était généré à partir des colonnes. Il n'apportait donc aucune information qu'un modèle tabulaire ne lit pas directement, et le LLM pondérait les caractéristiques moins finement qu'un modèle entraîné sur les 1 168 ventes.

**Délai et dépendance à un service externe.** Le RAG est aussi plus lourd à utiliser. Avant toute prédiction, chaque maison d'entraînement doit être décrite par un texte, encodée en vecteur et indexée dans la base vectorielle. Chaque prédiction demande ensuite un appel à l'API d'OpenAI, avec raisonnement : estimer les 292 maisons de validation a pris environ cinq minutes, avec huit appels en parallèle. XGBoost s'entraîne en quelques secondes et prédit toutes les maisons presque instantanément. Le RAG dépend en plus d'un service externe : chaque appel est facturé, demande une clé d'API et une connexion, et deux appels identiques peuvent donner des prix différents. Ce coût rendait aussi difficile une validation croisée, qui aurait multiplié le nombre d'appels.

### 1.4 Ensemble de cinq modèles

L'[expérience 04](experiences/04_ensemble.ipynb) combine cinq modèles de deux familles, entraînés sur les mêmes données préparées. Les cinq modèles sont :

| Modèle | Famille |
|---|---|
| Lasso | Linéaire |
| ElasticNet | Linéaire |
| Kernel Ridge | Linéaire, sur des données enrichies |
| Gradient Boosting | Arbres |
| XGBoost | Arbres |

Les modèles linéaires et les arbres lisent les données de façons différentes : ils devraient donc se tromper sur des maisons différentes, et leur moyenne compenser une partie de ces erreurs (section 2).

**Résultats.** En validation croisée sur les 1 168 ventes d'entraînement, la moyenne simple des cinq obtient une RMSE log de **0,1067**, contre 0,1104 pour le meilleur modèle seul, Kernel Ridge. Sur les 292 maisons de validation, la moyenne obtient **0,1233**, contre 0,1247 pour Kernel Ridge et 0,1257 pour XGBoost seul.

<p align="left"><img src="../docs/figures/04_correlation_erreurs_5_modeles.png" alt="Corrélation des erreurs hors pli des cinq modèles, expérience 04" width="500"></p>

**Lecture de la figure.** La matrice de corrélation des erreurs hors pli de l'expérience 04 montre deux familles très marquées : les trois modèles linéaires se trompent presque sur les mêmes maisons (corrélation de 0,97 à 1,00; Lasso et ElasticNet sont à 1,00), et les deux modèles d'arbres aussi (0,92). Entre les deux familles, la corrélation tombe à 0,83–0,85 : c'est là que la moyenne gagne. Mais garder trois modèles linéaires presque identiques est-il utile?

### 1.5 Ensemble de trois modèles

Si trois modèles linéaires se trompent sur les mêmes maisons, en garder trois est-il utile? L'[expérience 06](experiences/06_ensemble_trois_modeles.ipynb) teste une moyenne de trois modèles moins redondants : Lasso pour les modèles linéaires simples, Kernel Ridge pour les interactions, et XGBoost pour les arbres. Le protocole est celui de l'expérience 04, sans aucun changement; seule la liste des modèles moyennés change.

| | Validation croisée | Validation (292 maisons) |
|---|---|---|
| Moyenne des cinq | **0,1067** | 0,1233 |
| Moyenne des trois | 0,1074 | **0,1222** |

La moyenne des trois fait un peu moins bien en validation croisée, et ce dans chacun des cinq plis. En validation, l'ordre s'inverse, surtout parce que Gradient Boosting est le moins bon modèle seul sur ces 292 maisons; avec si peu de maisons, l'écart reste dans la marge d'incertitude.

### 1.6 Bilan des expériences

| Approche | Expérience | RMSE log, validation (292 maisons) |
|---|---|---|
| RAG + LLM, version 1 | 01 | 0,140 |
| RAG + LLM, version 2 | 02 | 0,133 |
| XGBoost réglé | 03 | 0,125 à 0,130 |
| Ensemble de cinq modèles | 04 | 0,1233 |
| Ensemble de trois modèles | 06 | 0,1222 |

Les ensembles font mieux que toutes les approches précédentes : **nous retenons donc un ensemble de modèles**. Les deux ensembles sont toutefois trop proches pour être départagés sur 292 maisons. La suite du notebook choisit quels modèles le composent, à partir de la validation croisée sur toutes les ventes (section 4).

### 1.7 Pourquoi une validation croisée à cinq plis?

Nous n'avons que 1 460 ventes pour entraîner et évaluer le modèle. Si nous gardons 20 % des ventes pour l'évaluation, le score est mesuré sur environ 292 maisons seulement, et il dépend beaucoup des maisons tirées au hasard. Pour atténuer ce hasard, nous utilisons une validation croisée à cinq plis : les ventes sont réparties en cinq groupes d'environ 292 maisons, appelés plis, et chaque groupe sert à tour de rôle à l'évaluation.

| Tour | Groupe 1 | Groupe 2 | Groupe 3 | Groupe 4 | Groupe 5 |
|---|---|---|---|---|---|
| 1 | **Évaluation** | Entraînement | Entraînement | Entraînement | Entraînement |
| 2 | Entraînement | **Évaluation** | Entraînement | Entraînement | Entraînement |
| 3 | Entraînement | Entraînement | **Évaluation** | Entraînement | Entraînement |
| 4 | Entraînement | Entraînement | Entraînement | **Évaluation** | Entraînement |
| 5 | Entraînement | Entraînement | Entraînement | Entraînement | **Évaluation** |

À chaque tour, un nouveau modèle est entraîné sur quatre groupes (environ 1 168 ventes), puis évalué sur le groupe restant. Le score final est la moyenne des cinq scores. Cette méthode a trois avantages :
- **chaque vente est évaluée une fois**, par un modèle qui ne l'a pas vue pendant son entraînement. Ces prédictions servent aussi à comparer les modèles et à tester leurs combinaisons (section 4);
- **le score dépend moins du hasard** : au tableau 4.2, le même ensemble obtient de 0,100 à 0,123 selon le groupe évalué. Une seule partition n'aurait donné qu'un de ces chiffres;
- **aucune vente n'est perdue** : toutes servent à la fois à l'entraînement et à l'évaluation.

Les cinq modèles entraînés pendant la validation croisée servent seulement à mesurer l'erreur. Le modèle final est ensuite entraîné une seule fois sur toutes les ventes (section 6). La répartition en groupes est aléatoire, mais toujours la même d'une exécution à l'autre (`random_state=42`), pour que les résultats soient reproductibles.

Contrairement aux expériences, qui préparaient les données à leur façon, la suite du notebook utilise le prétraitement de l'étape 4 (`src/preprocessing/`), adapté à chaque famille de modèles : les modèles linéaires le reçoivent au complet, les arbres sans le logarithme des colonnes à longue queue ni la standardisation, qui ne changent rien pour eux. Les scores diffèrent donc légèrement de ceux des expériences.

## 2. Pourquoi combiner des modèles

### 2.1 Pourquoi la moyenne réduit l'erreur

Pour estimer le prix d'une maison, on peut demander l'avis de plusieurs évaluateurs plutôt que d'un seul. Chacun se trompe un peu, mais pas toujours dans le même sens ni sur les mêmes maisons. En faisant la moyenne de leurs avis, une partie de ces erreurs s'annule. Cette idée a une condition : les évaluateurs doivent avoir des façons de raisonner différentes. Deux évaluateurs qui utilisent exactement la même méthode font les mêmes erreurs, et la moyenne de leurs avis n'apporte rien. C'est le cas de Lasso et d'ElasticNet (section 1.4). La section 4 vérifie ce que coûte ou rapporte le fait de les garder tous les deux.

Cette intuition se démontre. Le résultat suivant, appelé décomposition de l'ambiguïté (Krogh et Vedelsby, 1995), vaut pour toute moyenne de modèles de régression lorsque l'erreur est mesurée au carré, comme dans la RMSE. Pour chaque observation, ici chaque maison, l'erreur de la moyenne se décompose ainsi :

$$
\underbrace{(\bar{p} - y)^2}_{\text{erreur de la moyenne}}
= \underbrace{\frac{1}{k}\sum_{i=1}^{k} (p_i - y)^2}_{\text{erreur moyenne des modèles}}
- \underbrace{\frac{1}{k}\sum_{i=1}^{k} (p_i - \bar{p})^2}_{\text{désaccord entre les modèles}}
$$

où $y$ est le prix réel de la maison, $p_i$ le prix prédit par le modèle $i$, $\bar{p}$ la moyenne des prix prédits et $k$ le nombre de modèles. Nous l'appliquons au logarithme des prix, l'échelle de la métrique de Kaggle.

Autrement dit, **l'erreur de la moyenne est égale à l'erreur moyenne des modèles, moins leur désaccord.** Faire la moyenne retire donc à l'erreur exactement la valeur du désaccord. Comme le désaccord n'est jamais négatif, la moyenne se trompe toujours moins que les modèles en moyenne. À l'inverse, si les modèles prédisent tous la même chose, le désaccord est nul et la moyenne n'apporte rien.

Un modèle beaucoup moins bon que les autres modifie les deux termes de droite. Il augmente le désaccord, puisque ses prédictions s'écartent de celles des autres modèles, mais il augmente encore plus l'erreur moyenne des modèles. L'erreur de la moyenne peut alors devenir plus grande que celle des bons modèles pris seuls. Il vaut donc mieux combiner des modèles aussi bons les uns que les autres, mais qui se trompent sur des maisons différentes : le désaccord vient alors de vraies différences entre les modèles, et non d'un modèle qui se trompe beaucoup.

*Référence :* Krogh, A. et Vedelsby, J. (1995). « Neural Network Ensembles, Cross Validation, and Active Learning ». *Advances in Neural Information Processing Systems 7* (NIPS 1994), p. 231–238. [Lien](https://proceedings.neurips.cc/paper/1994/hash/b8c37e33defde51cf91e1e03e51657da-Abstract.html)

### 2.2 Les façons de combiner des modèles

Il existe plusieurs façons de combiner des modèles. Elles se divisent en deux groupes :
- **combiner des modèles différents**, déjà entraînés chacun de leur côté : moyenne simple, moyenne pondérée et empilement;
- **construire un ensemble à partir d'un seul type de modèle**, entraîné plusieurs fois : bagging et boosting.

Dans les exemples suivants, une maison est vendue 200 k\$, et trois modèles prédisent respectivement 190 k\$, 205 k\$ et 215 k\$.

#### Moyenne simple

Chaque modèle reçoit le même poids : on additionne les prédictions et on divise par le nombre de modèles. C'est la méthode la plus simple, car elle n'a aucun paramètre à apprendre. Elle suppose que les modèles sont à peu près aussi bons les uns que les autres.

**Exemple.** (190 + 205 + 215) / 3 = 203,3 k\$. Chaque modèle compte pour un tiers.

#### Moyenne pondérée

Chaque modèle reçoit un poids différent, et les poids additionnés donnent 1. Un modèle plus fiable compte davantage. Les poids sont appris sur des données de validation, en cherchant ceux qui donnent la plus petite erreur. Avec peu de données, ces poids risquent toutefois de s'ajuster au hasard de l'échantillon plutôt qu'à une vraie différence de qualité entre les modèles.

**Exemple.** Avec des poids de 0,5, 0,3 et 0,2 : 0,5 × 190 + 0,3 × 205 + 0,2 × 215 = 199,5 k\$. Le premier modèle compte ici pour la moitié de la prédiction.

#### Empilement (*stacking*)

Un second modèle, appelé méta-modèle, apprend comment combiner les prédictions des premiers. Il est entraîné sur les prédictions hors pli des modèles et sur les vrais prix. Contrairement à la moyenne pondérée, il peut apprendre des règles plus fines, par exemple faire davantage confiance à un modèle pour les maisons chères et à un autre pour les maisons bon marché. C'est la méthode la plus souple, mais aussi la plus complexe et la plus exposée au surapprentissage.

**Exemple.** Le méta-modèle reçoit les trois prédictions (190, 205 et 215 k\$) comme données d'entrée. S'il a appris que le troisième modèle est le plus fiable pour les grandes maisons, il peut prédire 212 k\$ pour une grande maison, mais suivre davantage les deux autres modèles pour une petite.

#### Bagging

Le même type de modèle est entraîné plusieurs fois, chaque fois sur un échantillon différent des ventes. Chaque échantillon est tiré au hasard avec remise : certaines ventes y apparaissent plusieurs fois, d'autres pas du tout. On fait ensuite la moyenne des prédictions. Cette méthode stabilise les modèles très sensibles aux données d'entraînement, comme les arbres de décision : un arbre seul change beaucoup si on retire quelques ventes, mais la moyenne de plusieurs centaines d'arbres change peu.

**Exemple.** Une forêt aléatoire entraîne 500 arbres de décision, chacun sur un échantillon différent de ventes. Pour la maison de 200 k\$, les arbres prédisent des prix variés, par exemple de 170 k\$ à 230 k\$, et la forêt renvoie leur moyenne.

#### Boosting

Les modèles sont entraînés l'un après l'autre, et chacun apprend à corriger les erreurs des précédents. La prédiction finale est la somme de toutes les corrections. Pour éviter de corriger trop vite, chaque correction est réduite par un taux d'apprentissage. Gradient Boosting et XGBoost fonctionnent ainsi : ce sont déjà des ensembles d'arbres. Notre configuration de XGBoost en compte 2 000.

**Exemple.** Un premier arbre prédit 180 k\$. Il reste une erreur de 20 k\$, et un deuxième arbre apprend à la corriger : il ajoute 12 k\$, pour un total de 192 k\$. Un troisième arbre corrige l'erreur restante de 8 k\$, et ainsi de suite, jusqu'à s'approcher de 200 k\$.

### 2.3 Notre choix : une moyenne simple sur le logarithme du prix

#### Une moyenne simple de cinq modèles

Parmi les méthodes de la section 2.2, nous retenons la moyenne simple : le prix prédit est la moyenne des prédictions de Lasso, d'ElasticNet, de Kernel Ridge, de Gradient Boosting et de XGBoost, avec le même poids pour chacun. C'est la méthode la plus simple, et elle convient à notre situation. Nos cinq modèles obtiennent des scores proches : aucun ne mérite plus de poids que les autres. De plus, avec environ 1 160 ventes pour l'entraînement, des poids appris risqueraient de suivre le hasard de l'échantillon plutôt qu'une vraie différence entre les modèles. L'expérience 04 l'a vérifié : une moyenne pondérée, avec des poids appris sur les prédictions hors pli, n'apportait rien (0,1068 contre 0,1067).

#### Une moyenne sur le logarithme du prix

Les prix de vente sont asymétriques : la plupart des maisons se vendent autour de 160 k\$, mais quelques-unes dépassent 500 k\$ (étape 3). Plutôt que de travailler directement sur les prix, les modèles travaillent sur leur logarithme, dont la distribution est beaucoup plus régulière.

Le logarithme change aussi la façon de compter les erreurs : il mesure des écarts en pourcentage plutôt qu'en dollars. Une erreur de 10 k\$ sur une maison de 100 k\$ compte autant qu'une erreur de 50 k\$ sur une maison de 500 k\$, puisque les deux représentent 10 % du prix. Les maisons chères ne dominent donc plus le calcul de l'erreur. C'est aussi la façon dont Kaggle évalue les prédictions.

Nous faisons donc la moyenne des trois logarithmes prédits, puis nous revenons en dollars. La moyenne est ainsi calculée sur la même échelle que celle des modèles et de la métrique.

### 2.4 Ce que la combinaison ne corrige pas

#### Les erreurs communes à tous les modèles

La moyenne corrige seulement les erreurs sur lesquelles les modèles ne sont pas d'accord. Si tous les modèles se trompent de la même façon sur une maison, la moyenne se trompe aussi.

Cela arrive surtout quand une information importante manque dans les données, par exemple un intérieur rénové ou une vente faite sous le prix du marché. Aucun modèle ne voit cette information, donc la moyenne ne peut pas la corriger.

#### La qualité de chaque modèle reste essentielle

La moyenne ne peut pas faire beaucoup mieux que les modèles qui la composent. Elle réduit l'erreur moyenne des modèles d'une valeur égale à leur désaccord, mais elle part de cette erreur moyenne (section 2.1). Si les modèles sont tous faibles, leur moyenne restera faible, même s'ils sont en désaccord.

Combiner des modèles ne remplace donc ni un bon prétraitement des données (étape 4) ni un bon choix de modèles. La combinaison est une dernière amélioration, qui s'ajoute à des modèles déjà bons. C'est pourquoi la section 3 présente chacun des cinq modèles retenus, ce qu'il capte dans nos données et pourquoi il est adapté à notre problème.

## 3. Les cinq modèles retenus

Nous retenons cinq modèles de deux familles, qui lisent les données de façons différentes. Lasso, ElasticNet et Kernel Ridge sont des modèles linéaires : ils supposent que le prix se construit en additionnant des effets. Gradient Boosting et XGBoost sont des modèles d'arbres de décision : ils découpent les maisons en groupes selon des seuils. Les cinq obtiennent des scores proches (tableau 4.3), mais un modèle linéaire et un modèle d'arbres ne se trompent pas sur les mêmes maisons, ce qui est la condition pour que leur moyenne soit utile (section 2).

Lasso, Kernel Ridge et XGBoost sont présentés en détail. ElasticNet et Gradient Boosting sont des variantes proches de Lasso et de XGBoost : nous présentons seulement ce qui les en distingue.

### 3.1 Lasso

**Principe.** Lasso est une régression linéaire : le logarithme du prix est prédit comme une somme pondérée des caractéristiques de la maison,

$$
\hat{y} = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \dots + \beta_p x_p
$$

où chaque $x_j$ est une caractéristique (la surface, la qualité, l'âge…) et chaque $\beta_j$ est son coefficient, c'est-à-dire le poids qu'elle a dans le prix.

Les coefficients sont ceux qui rendent la fonction suivante la plus petite possible :

$$
\hat{\beta} = \arg\min_{\beta} \;
\underbrace{\frac{1}{2n} \sum_{i=1}^{n} \left( y_i - \hat{y}_i \right)^2}_{\text{erreur sur les ventes d'entraînement}}
+ \underbrace{\alpha \sum_{j=1}^{p} |\beta_j|}_{\text{pénalité sur les coefficients}}
$$

où $n$ est le nombre de ventes d'entraînement, $y_i$ le logarithme du prix réel de la vente $i$, $\hat{y}_i$ sa prédiction et $\alpha$ l'intensité de la pénalité.

Le premier terme est l'erreur du modèle sur les ventes d'entraînement. Le second est une pénalité qui augmente avec la taille des coefficients. En cherchant à rendre la somme des deux la plus petite possible, Lasso conserve seulement les caractéristiques vraiment utiles et met le coefficient des autres à zéro. Le paramètre $\alpha$ règle la force de cette pénalité : plus il est grand, plus Lasso retire de caractéristiques.

<p align="left">
  <img src="../docs/figures/03_1_lasso_coefficients.png" alt="Coefficients d'une régression linéaire ordinaire et de Lasso" width="900">
</p>

**Lecture de la figure.** Dans cet exemple à dix caractéristiques, la régression ordinaire attribue un coefficient à chacune d'elles, même à celles qui apportent peu. Lasso, lui, n'en conserve que cinq : il retire les caractéristiques selon leur utilité. Les salles de bain et les chambres en sont un exemple. La régression ordinaire leur donne un petit coefficient négatif, un résultat contre-intuitif qui s'explique par le fait qu'elles répètent une information déjà donnée par la surface habitable. Comme leur effet propre est faible, Lasso met leur coefficient à zéro.

**Adéquation à nos données.** Le prix d'une maison s'explique en grande partie par des effets qui s'additionnent : la qualité, la surface, l'âge et l'emplacement. Nos données contiennent toutefois beaucoup de colonnes, souvent liées entre elles, pour un nombre limité de ventes. Dans ce contexte, une régression ordinaire risque d'apprendre les particularités des ventes d'entraînement plutôt que les tendances générales du marché, en donnant du poids à de petits effets qui n'existent que par hasard dans l'échantillon. Grâce à sa pénalité, Lasso écarte ces petits effets et ne conserve que les caractéristiques qui expliquent vraiment le prix.

**Contribution à l'ensemble.** Lasso est le modèle le plus simple et le plus stable des cinq. Il capte les grandes tendances du prix et peut les prolonger au-delà des ventes connues. Par exemple, pour une maison plus grande que toutes celles de l'entraînement, il prédit un prix plus élevé que tous ceux qu'il a vus, alors que les modèles d'arbres restent limités aux prix observés pendant l'entraînement (section 3.5).

**Limites.** Lasso suppose que chaque caractéristique a le même effet sur toutes les maisons. Par exemple, une salle de bain de plus ajoute le même pourcentage au prix d'une petite maison et d'une grande. Il ne tient pas compte des relations entre les caractéristiques, comme une surface qui vaut plus dans une maison de grande qualité, ni des effets de seuil. Kernel Ridge et les modèles d'arbres compensent ces limites.

### 3.2 ElasticNet

**Principe.** ElasticNet est une régression linéaire, comme Lasso, mais sa pénalité combine celle de Lasso (valeur absolue des coefficients) et celle de Ridge (carré des coefficients) :

$$
\hat{\beta} = \arg\min_{\beta} \;
\underbrace{\frac{1}{2n} \sum_{i=1}^{n} \left( y_i - \hat{y}_i \right)^2}_{\text{erreur sur les ventes d'entraînement}}
+ \underbrace{\alpha \, \rho \sum_{j=1}^{p} |\beta_j|}_{\text{pénalité de Lasso}}
+ \underbrace{\frac{\alpha \, (1 - \rho)}{2} \sum_{j=1}^{p} \beta_j^2}_{\text{pénalité de Ridge}}
$$

où $\rho$, entre 0 et 1, fixe la part de chaque pénalité. Avec $\rho = 1$, on retrouve Lasso.

**Adéquation à nos données.** Nos données contiennent des groupes de colonnes très liées : la surface habitable, la surface du premier étage et la surface totale, ou le nombre de places et la surface du garage. Face à un tel groupe, Lasso a tendance à garder une seule colonne et à mettre les autres à zéro, un choix qui peut changer d'un échantillon à l'autre. La part Ridge de la pénalité pousse plutôt ElasticNet à partager le poids entre les colonnes du groupe, ce qui rend ses coefficients plus stables.

**Contribution à l'ensemble.** En pratique, ElasticNet se trompe exactement sur les mêmes maisons que Lasso (corrélation de 1,00, section 1.4) : sur nos données, les deux modèles sont presque identiques. Le garder revient à donner un poids double à l'approche de Lasso dans la moyenne. La section 4 mesure ce que ce choix coûte ou rapporte.

*Référence :* Zou, H. et Hastie, T. (2005). « Regularization and Variable Selection via the Elastic Net ». *Journal of the Royal Statistical Society, Series B*, 67(2), p. 301–320.

### 3.3 Kernel Ridge

**Principe.** Kernel Ridge est une régression Ridge appliquée à des données enrichies. Il sert à capter les interactions entre les caractéristiques, c'est-à-dire les cas où l'effet d'une caractéristique dépend d'une autre : par exemple, un pied carré de plus peut valoir davantage dans une maison de grande qualité. Pour cela, le modèle ajoute aux caractéristiques de la maison leurs carrés et leurs produits deux à deux. Le logarithme du prix est alors prédit ainsi :

$$
\hat{y} =
\underbrace{\beta_0}_{\text{valeur de départ}}
+ \underbrace{\sum_{j} \beta_j x_j}_{\text{effet de chaque caractéristique}}
+ \underbrace{\sum_{j} \beta_{jj}\, x_j^2}_{\text{carrés}}
+ \underbrace{\sum_{j<k} \beta_{jk}\, x_j x_k}_{\text{interactions}}
$$

où chaque $x_j$ est une caractéristique de la maison et chaque $\beta$ est un coefficient appris pendant l'entraînement.

- $\beta_0$ est la valeur de départ de la prédiction, comme l'ordonnée à l'origine d'une droite. Les autres termes viennent l'augmenter ou la diminuer selon les caractéristiques de la maison.
- Le premier somme est la même que dans Lasso : chaque caractéristique, multipliée par son coefficient, ajoute un effet fixe au prix.
- Les carrés $x_j^2$ permettent à l'effet d'une caractéristique de changer selon sa propre valeur. Par exemple, un pied carré de plus peut rapporter moins dans une maison déjà très grande que dans une petite maison.
- Les produits $x_j x_k$ créent les interactions : ils permettent à l'effet d'une caractéristique de dépendre d'une autre.

Il reste à trouver la valeur de tous ces coefficients. Comme pour Lasso, le modèle cherche un compromis entre deux objectifs : bien prédire les ventes d'entraînement, et garder des coefficients petits pour ne pas s'ajuster au hasard de l'échantillon. Il choisit donc les coefficients qui rendent la fonction suivante la plus petite possible :

$$
\hat{\beta} = \arg\min_{\beta} \;
\underbrace{\sum_{i=1}^{n} \left( y_i - \hat{y}_i \right)^2}_{\text{erreur sur les ventes d'entraînement}}
+ \underbrace{\alpha \sum \beta^2}_{\text{pénalité sur les coefficients}}
$$

où $n$ est le nombre de ventes d'entraînement, $y_i$ le logarithme du prix réel de la vente $i$, $\hat{y}_i$ sa prédiction et $\alpha$ l'intensité de la pénalité.

Le premier terme est l'erreur du modèle sur les ventes d'entraînement, comme pour Lasso. La différence est dans la pénalité : Ridge pénalise le carré des coefficients plutôt que leur valeur absolue. Un petit coefficient ne coûte alors presque rien, si bien que Ridge réduit tous les coefficients sans en mettre aucun à zéro. Chaque interaction garde ainsi un petit poids, à moins que les données justifient un poids plus grand.

**Le noyau.** Avec nos 227 caractéristiques, les carrés et les produits deux à deux donneraient environ 26 000 colonnes, soit beaucoup plus que nos 1 460 ventes. Le noyau évite de construire ces colonnes.

Pour s'entraîner et prédire, Ridge n'a besoin que de comparer les maisons deux à deux, à l'aide de leur produit scalaire $x \cdot z = x_1 z_1 + x_2 z_2 + \dots + x_p z_p$. Un **noyau** est une fonction $K(x, z)$ qui donne directement le produit scalaire des données enrichies, à partir des caractéristiques d'origine seulement. On remplace donc $x \cdot z$ par $K(x, z)$, et le modèle travaille sur les données enrichies sans jamais les construire. C'est ce qu'on appelle l'astuce du noyau (*kernel trick*).

**Exemple de calcul.** Prenons deux maisons décrites par deux caractéristiques, $x = (2, 1)$ et $z = (1, 3)$, et le noyau $K(x, z) = (x \cdot z)^2$.

- *Sans le noyau*, on construit d'abord les produits deux à deux de chaque maison, $(x_1^2,\ x_1 x_2,\ x_2 x_1,\ x_2^2)$. On obtient $(4, 2, 2, 1)$ pour $x$ et $(1, 3, 3, 9)$ pour $z$, dont le produit scalaire vaut $4 \times 1 + 2 \times 3 + 2 \times 3 + 1 \times 9 = 25$.
- *Avec le noyau*, on calcule seulement $x \cdot z = 2 \times 1 + 1 \times 3 = 5$, puis $5^2 = 25$.

Le résultat est le même, mais le second calcul ne construit aucune nouvelle colonne. Avec nos 227 caractéristiques, l'écart devient important : d'un côté, environ 26 000 colonnes à construire pour chaque maison; de l'autre, 227 multiplications et un carré.

Notre modèle utilise le noyau $K(x, z) = (\gamma \, x \cdot z + c)^2$, où $\gamma$ et $c$ sont deux paramètres fixés à l'avance. En développant ce carré, on obtient $\gamma^2 (x \cdot z)^2 + 2c\gamma \, (x \cdot z) + c^2$. Le premier terme correspond aux produits deux à deux, le deuxième aux caractéristiques d'origine et le dernier à une constante : on retrouve les données enrichies décrites plus haut.

<p align="left"><img src="../docs/figures/03_2_kernel_ridge_interaction.png" alt="Prix prédit selon la surface et la qualité : modèle linéaire et Kernel Ridge" width="1000"></p>

**Lecture de la figure.** Dans cet exemple, les deux modèles sont entraînés sur deux caractéristiques seulement : la surface habitable et la qualité globale. On compare deux maisons de 1 500 pi², identiques sauf leur qualité : l'une est de qualité moyenne (5), l'autre de très bonne qualité (8). On ajoute ensuite 500 pi² à chacune et on mesure la hausse du prix prédit.

Le modèle linéaire donne la même hausse aux deux maisons : +14 %. Dans sa formule, la surface n'apparaît que dans un seul terme, $\beta_1 x_1$, qui ne dépend pas de la qualité. Agrandir une maison a donc le même effet, quelle que soit sa qualité.

Kernel Ridge donne une hausse de 11 % à la maison de qualité moyenne, mais de 18 % à celle de très bonne qualité. Comme les deux maisons ont la même surface, le carré de la surface joue de la même façon pour les deux. La différence vient du produit surface × qualité : grâce à ce terme, l'effet de la surface change avec la qualité. Selon ces ventes, agrandir une maison rapporte donc davantage quand elle est de meilleure qualité.

**Adéquation à nos données.** Le prix d'une maison s'explique en grande partie par des effets qui s'additionnent, ce qui explique les bons résultats des modèles linéaires. Parfois, toutefois, l'effet d'une caractéristique dépend d'une autre, comme la surface et la qualité dans la figure. Kernel Ridge peut capter ces relations sans qu'on ait à choisir à l'avance quelles paires de caractéristiques comptent. En contrepartie, il a beaucoup de coefficients à apprendre pour un nombre limité de ventes, et il risquerait d'apprendre les particularités des ventes d'entraînement plutôt que les tendances du marché. Sa pénalité limite ce risque : elle garde chaque coefficient petit, sauf si les données justifient un effet plus fort.

**Contribution à l'ensemble.** Kernel Ridge est le meilleur modèle seul (tableau 4.3). Comme il appartient à la même famille que Lasso, leurs erreurs sont très liées (0,97, section 1.4). Il ne se trompe toutefois pas exactement sur les mêmes maisons, pour deux raisons : il tient compte des interactions, que Lasso ne voit pas, et il garde toutes les caractéristiques avec un petit poids, alors que Lasso en retire une partie.

**Limites.** Kernel Ridge est difficile à interpréter. Ses effets sont répartis entre des milliers de carrés et de produits, et le modèle ne calcule même jamais ces coefficients, puisqu'il ne construit pas les colonnes. On ne peut donc pas lire, comme avec Lasso, ce que vaut un pied carré ou un point de qualité.

Il est aussi sensible à l'échelle des colonnes. Comme le noyau multiplie les caractéristiques entre elles, une colonne aux grandes valeurs peut dominer toutes les autres. C'est pourquoi les colonnes sont standardisées à l'étape 4. Dans une de nos expériences (`notebooks/experiences/04_ensemble.ipynb`), une seule colonne mal mise à l'échelle avait suffi à faire passer son erreur de 0,11 à 0,17.

Enfin, il ne capte que les interactions entre deux caractéristiques, et ses effets restent réguliers : il ne repère pas les effets de seuil, comme un prix qui change brusquement à partir d'une certaine qualité. Les modèles d'arbres compensent cette limite (section 3.5). De plus, comme le modèle compare les ventes deux à deux, son calcul reste rapide avec nos 1 460 ventes, mais deviendrait lourd avec des centaines de milliers de ventes.

*Référence :* Saunders, C., Gammerman, A. et Vovk, V. (1998). « Ridge Regression Learning Algorithm in Dual Variables ». *Proceedings of the 15th International Conference on Machine Learning (ICML '98)*, p. 515–521.

### 3.4 Gradient Boosting

**Principe.** Gradient Boosting (`GradientBoostingRegressor` de scikit-learn) repose sur le même principe que XGBoost (section 3.5) : une suite d'arbres de décision, chacun corrigeant une petite partie de l'erreur laissée par les précédents. Il s'en distingue par trois choix :

- **La perte de Huber.** Chaque nouvel arbre corrige une erreur mesurée par la perte de Huber plutôt que par le carré de l'écart :

$$
L_\delta(r) =
\begin{cases}
\frac{1}{2} r^2 & \text{si } |r| \le \delta \\
\delta \left( |r| - \frac{1}{2} \delta \right) & \text{sinon}
\end{cases}
$$

où $r$ est l'écart entre le logarithme du prix réel et sa prédiction. Les petits écarts comptent au carré, comme dans la RMSE; au-delà du seuil $\delta$, ils ne comptent plus que linéairement. Le seuil $\delta$ est recalculé à chaque arbre : c'est le 90ᵉ centile des écarts absolus (valeur par défaut de scikit-learn).
- **Le tirage des colonnes à chaque division.** À chaque question posée dans un arbre, seule une partie des colonnes, environ la racine carrée de leur nombre, est examinée (`max_features="sqrt"`).
- **Des feuilles d'au moins 15 ventes**, au lieu d'une pénalité sur la valeur des feuilles comme dans XGBoost.

**Adéquation à nos données.** Même sans les ventes 524 et 1299, quelques ventes ont un prix très éloigné de ce que leurs caractéristiques laissent prévoir, comme les ventes partielles ou anormales. Avec une perte au carré, ces ventes tireraient chaque arbre vers elles. La perte de Huber réduit leur poids, et les arbres apprennent surtout des ventes ordinaires.

**Contribution à l'ensemble.** Gradient Boosting est le second modèle d'arbres. Ses erreurs sont proches de celles de XGBoost (0,92, section 1.4), mais pas identiques : ses arbres sont construits autrement, et sa perte de Huber le rend moins sensible aux prix extrêmes. L'expérience 06 (section 1.5) a montré que le retirer de la moyenne la rendait un peu moins bonne.

**Limites.** C'est le modèle le plus long à entraîner (3 000 arbres) et, comme XGBoost, il ne prolonge pas les tendances au-delà des ventes vues à l'entraînement.

*Références :*
- Friedman, J. H. (2001). « Greedy Function Approximation: A Gradient Boosting Machine ». *The Annals of Statistics*, 29(5), p. 1189–1232.
- Huber, P. J. (1964). « Robust Estimation of a Location Parameter ». *The Annals of Mathematical Statistics*, 35(1), p. 73–101.

### 3.5 XGBoost

**Principe.** XGBoost prédit le prix à l'aide d'arbres de décision. Un arbre pose une suite de questions par oui ou par non sur les caractéristiques de la maison, par exemple « la qualité est-elle d'au moins 7? » puis « la surface dépasse-t-elle 1 400 pi²? ». Chaque maison aboutit ainsi dans une feuille de l'arbre, qui lui attribue une valeur.

Pour bien apprendre, XGBoost construit plusieurs arbres, l'un après l'autre. Chaque nouvel arbre corrige l'erreur laissée par les arbres précédents : c'est ce qu'on appelle le boosting (section 2.2). Les arbres déjà construits ne sont jamais modifiés, et leurs corrections s'additionnent. Le logarithme du prix est prédit ainsi :

$$
\hat{y} =
\underbrace{f_0}_{\text{valeur de départ}}
+ \underbrace{\eta \sum_{t=1}^{T} f_t(x)}_{\text{corrections des arbres}}
$$

où $f_0$ est la même valeur de départ pour toutes les maisons, $f_t(x)$ la valeur de la feuille où tombe la maison $x$ dans l'arbre $t$, $T$ le nombre d'arbres et $\eta$ le taux d'apprentissage. Le taux d'apprentissage est un petit nombre qui réduit chaque correction : chaque arbre ne corrige qu'une partie de l'erreur. Le modèle avance ainsi à petits pas, ce qui lui évite de coller trop vite aux ventes d'entraînement, mais demande beaucoup d'arbres.

Pour construire chaque nouvel arbre, XGBoost calcule d'abord l'erreur qui reste pour chaque vente :

$$
r_i = y_i - \hat{y}_i^{(t-1)}
$$

où $y_i$ est le logarithme du prix réel de la vente $i$ et $\hat{y}_i^{(t-1)}$ sa prédiction par les arbres déjà construits. Par exemple, si une maison s'est vendue 200 k\$ et que les arbres précédents prédisent 180 k\$, il reste une erreur de 20 k\$ à corriger (calculée en réalité sur le logarithme du prix). Le nouvel arbre $f_t$ est ensuite choisi pour rendre la fonction suivante la plus petite possible :

$$
f_t = \arg\min_{f} \;
\underbrace{\sum_{i=1}^{n} \left( r_i - f(x_i) \right)^2}_{\text{erreur sur l'écart à corriger}}
+ \underbrace{\lambda \sum_{j} w_j^2}_{\text{pénalité sur les feuilles}}
$$

où $w_j$ est la valeur de la feuille $j$ et $\lambda$ l'intensité de la pénalité.

Le premier terme montre que le nouvel arbre ne cherche pas à prédire le prix lui-même, mais l'erreur $r_i$ laissée par les arbres précédents. Le second terme limite la valeur des feuilles. Sans pénalité, une feuille donnerait la moyenne des erreurs de ses ventes. Avec la pénalité, on divise par le nombre de ventes plus $\lambda$, plutôt que par le nombre de ventes seul. Pour une feuille de plusieurs centaines de ventes, cet ajout ne change presque rien. Pour une feuille de quelques ventes seulement, il réduit nettement la correction. La pénalité freine donc surtout les petits groupes, dont la moyenne est peu fiable.

Enfin, chaque arbre est construit sur une partie seulement des données : XGBoost tire au hasard une partie des ventes et une partie des colonnes, et refait ce tirage pour chaque nouvel arbre. Ainsi, une vente inhabituelle ne pèse pas sur toutes les corrections. De même, une colonne très utile, comme la qualité, est absente de certains arbres, qui doivent alors apprendre avec d'autres colonnes, comme la surface ou le quartier. Le modèle tire ainsi son information d'un plus grand nombre de colonnes et dépend moins d'une seule d'entre elles.

<p align="left"><img src="../docs/figures/03_3_xgboost_arbres.png" alt="Les deux premiers arbres d'un XGBoost simplifié : chaque arbre corrige l'erreur qui reste" width="900"></p>

**Lecture de la figure.** Cet exemple simplifié montre les deux premiers arbres. Pour qu'ils restent lisibles, chaque arbre ne reçoit ici que deux colonnes sur quatre et ne pose que deux questions de suite. Dans notre modèle, le principe est le même, mais chaque arbre reçoit beaucoup plus de colonnes, tirées parmi toutes celles des données, et peut poser plus de questions. Le tirage des colonnes a été choisi pour l'exemple.

L'arbre 1 reçoit la qualité et la surface. Il sépare les ventes en quatre groupes selon ces deux colonnes : les maisons de qualité 7 ou moins et de petite surface se vendent en moyenne 27 % sous la prédiction de départ, alors que celles de qualité 8 et plus se vendent 76 % au-dessus. L'arbre n'ajoute toutefois qu'une petite partie de cet écart : −0,64 % pour le premier groupe et +1,13 % pour le dernier.

L'arbre 2 part ensuite de l'erreur qui reste après l'arbre 1. Comme il a reçu d'autres colonnes, le garage et l'année de construction, il pose d'autres questions et forme d'autres groupes. Il ajoute à son tour sa propre petite correction. Les écarts restent grands d'un arbre à l'autre, puisque chaque arbre n'en corrige qu'une petite partie. La prédiction finale est la somme de toutes ces corrections.

**Adéquation à nos données.** Nos données sont présentées en tableau : chaque vente est décrite par des surfaces, des scores de qualité, des années et des catégories. Sur ce type de données, les modèles d'arbres restent parmi les plus performants (Grinsztajn et al., 2022). Ils repèrent d'eux-mêmes les seuils, par exemple un prix qui augmente nettement à partir d'une certaine qualité. Ils combinent aussi les caractéristiques d'une question à l'autre, comme l'arbre 1 de la figure, qui interroge la qualité puis la surface. Tout cela se fait sans qu'on ait à préciser à l'avance la forme de la relation entre les caractéristiques et le prix. Enfin, comme un arbre ne fait que comparer des valeurs à des seuils, il ne dépend pas de l'échelle des colonnes.

**Contribution à l'ensemble.** Lasso, ElasticNet et Kernel Ridge sont des modèles linéaires : ils lisent le prix comme une somme d'effets. XGBoost et Gradient Boosting lisent les données autrement, comme une suite de seuils. Leurs erreurs ressemblent donc moins à celles des modèles linéaires (0,83 à 0,85, contre 0,97 à 1,00 entre modèles linéaires, section 1.4). Même si leurs scores seuls sont les moins bons des cinq, ce sont donc eux qui devraient apporter le plus de désaccord, et donc le plus grand gain à la moyenne (section 2.1). La section 4 le vérifie sur les données.

**Limites.** XGBoost ne prolonge pas les tendances au-delà des maisons vues à l'entraînement. Dans chaque arbre, une maison plus grande que toutes les autres tombe dans la même feuille que les plus grandes maisons connues. Elle reçoit donc à peu près le même prix qu'elles, alors qu'un modèle linéaire comme Lasso prédirait un prix plus élevé (section 3.1). Pour la même raison, ses prédictions changent par paliers, à chaque seuil, plutôt que de façon régulière.

XGBoost demande aussi de régler plusieurs paramètres : le nombre d'arbres, leur profondeur, le taux d'apprentissage, et la part des ventes et des colonnes tirées pour chaque arbre. Un mauvais réglage peut lui faire apprendre les particularités des ventes d'entraînement, par exemple avec un taux d'apprentissage trop grand ou des arbres trop profonds.

Enfin, ses prédictions sont difficiles à expliquer. Un seul arbre se lit facilement, comme dans la figure, mais la prédiction finale additionne les corrections d'un très grand nombre d'arbres. On ne peut donc pas dire simplement ce que vaut un pied carré ou un point de qualité.

*Références :*
- Chen, T. et Guestrin, C. (2016). « XGBoost: A Scalable Tree Boosting System ». *Proceedings of the 22nd ACM SIGKDD International Conference on Knowledge Discovery and Data Mining*, p. 785–794.
- Grinsztajn, L., Oyallon, E. et Varoquaux, G. (2022). « Why Do Tree-Based Models Still Outperform Deep Learning on Typical Tabular Data? ». *Advances in Neural Information Processing Systems 35 (NeurIPS 2022)*.

### 3.6 Tableau comparatif

| | Lasso | ElasticNet | Kernel Ridge | Gradient Boosting | XGBoost |
|---|---|---|---|---|---|
| **Famille** | Linéaire | Linéaire | Linéaire, sur des données enrichies | Arbres de décision | Arbres de décision |
| **Comment il lit les données** | Une somme d'effets | Une somme d'effets | Une somme d'effets, avec les carrés et les interactions deux à deux | Une suite de questions sur des seuils | Une suite de questions sur des seuils |
| **Caractéristiques utilisées** | Seulement les plus utiles | Les plus utiles, avec le poids partagé entre colonnes liées | Toutes, avec leurs carrés et leurs produits | Une partie, tirée au hasard à chaque division | Une partie, tirée au hasard pour chaque arbre |
| **Pénalité** | Valeur absolue des coefficients : en met certains à zéro | Valeur absolue et carré des coefficients | Carré des coefficients : les réduit tous | Petits pas, feuilles d'au moins 15 ventes, perte de Huber | Petits pas, pénalité sur les feuilles et tirage au hasard |
| **Point fort** | Simple, stable et facile à interpréter | Stable face aux colonnes liées | Capte les interactions entre caractéristiques | Capte les seuils; peu sensible aux prix extrêmes | Capte les seuils, sans supposer de forme |
| **Prolonge les tendances** | Oui | Oui | Oui | Non | Non |
| **Sensible à l'échelle des colonnes** | Oui | Oui | Oui, fortement | Non | Non |
| **Limite principale** | Ne voit ni les interactions ni les seuils | Presque identique à Lasso sur nos données | Difficile à interpréter; ne voit pas les seuils | Long à entraîner; prédictions par paliers | Prédictions par paliers; plusieurs réglages; difficile à interpréter |
| **Score seul, RMSE log (tableau 4.3)** | 0,1111 | 0,1112 | 0,1100 | 0,1138 | 0,1132 |

Les cinq modèles jouent des rôles complémentaires. Lasso est le modèle de base : il garde seulement les caractéristiques importantes, et ses coefficients se lisent directement. ElasticNet en est une variante plus stable face aux colonnes liées, mais presque identique sur nos données. Kernel Ridge part de la même idée et va plus loin : il ajoute un très grand nombre d'interactions que Lasso ne voit pas. Gradient Boosting et XGBoost, enfin, ajoutent les seuils qu'aucun modèle linéaire ne repère, alors que les modèles linéaires prolongent les tendances là où les arbres s'arrêtent. La section 4 vérifie que cette combinaison se traduit par une meilleure moyenne.

## 4. La preuve par les données

Cette section vérifie sur nos données si les arguments des sections 2 et 3 tiennent : combiner des modèles qui se trompent sur des maisons différentes devrait donner une meilleure moyenne, et il faut choisir quels modèles garder parmi les cinq.

Pour le vérifier, nous entraînons d'abord les cinq modèles par validation croisée à cinq plis (section 1.7), avec le prétraitement de l'étape 4 adapté à chaque famille. Avec cinq modèles, on peut former 31 combinaisons : 5 modèles seuls, 10 paires, 10 trios, 5 groupes de quatre et les cinq ensemble. Pour chaque combinaison, nous faisons la moyenne des prédictions de ses modèles, puis nous calculons sa RMSE log sur les ventes typiques, en moyenne sur les cinq plis, comme à la section 1.7. Aucun nouvel entraînement n'est nécessaire : chaque modèle étant entraîné de son côté, la moyenne d'un groupe de modèles donne exactement la même prédiction que si ce groupe avait été entraîné seul.

Le tableau 4.1 présente les 10 meilleures combinaisons. Le tableau 4.2 compare, pli par pli, les cinq modèles aux combinaisons qui servent de référence : les trois modèles de l'expérience 06, la même combinaison avec Gradient Boosting au lieu de XGBoost, les cinq sans ElasticNet et le meilleur modèle seul. Le tableau 4.3 donne le score de chaque modèle seul, pli par pli.

In [1]:
from html import escape
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import HTML, Markdown, display
from sklearn.model_selection import KFold

from src.evaluation import cv_rmse
from src.training import (
    MEMBERS, RANDOM_STATE, cross_validate_model, fit_model, load_data, make_model,
)

def show_title(title):
    display(Markdown(f"**{title}**"))

# =======================================================================
# Chargement des données
# =======================================================================
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
X, y, test = load_data(root / "data/train.csv", root / "data/test.csv")
print(f"{len(X)} ventes étiquetées dans train.csv, {len(test)} maisons dans test.csv.")

# =======================================================================
# Validation croisée des cinq modèles et de leur moyenne (section 1.7)
# =======================================================================
folds = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
oof, _ = cross_validate_model(make_model(list(MEMBERS)), X, y, folds)
FOLDS = [f"Pli {k}" for k in range(1, folds.get_n_splits() + 1)]

# =======================================================================
# Moyenne des log des prix hors pli pour chaque combinaison (sans réentraîner)
# =======================================================================
LINEAR = {"Lasso", "ElasticNet", "Kernel Ridge"}
log_oof = np.log(oof[list(MEMBERS)])

def fold_rmse(names):
    # RMSE log de chaque pli pour la moyenne des log des modèles donnés.
    pred = np.exp(log_oof[list(names)].mean(axis=1))
    return [cv_rmse(y.iloc[v], pred.iloc[v], X.iloc[v]) for _, v in folds.split(X)]

rows = []
for size in range(1, len(MEMBERS) + 1):
    for names in combinations(MEMBERS, size):
        scores = fold_rmse(names)
        linear = sum(name in LINEAR for name in names)
        family = "Linéaires" if linear == size else "Arbres" if linear == 0 else "Les deux"
        rows.append({"Combinaison": " + ".join(names), "Nombre de modèles": size,
                     "Familles": family, **dict(zip(FOLDS, scores)),
                     "RMSE log (moyenne des plis)": np.mean(scores),
                     "Écart type entre plis": np.std(scores)})

# Tri par RMSE moyenne croissante; le rang 1 est la meilleure des 31 combinaisons
combos = (pd.DataFrame(rows).sort_values("RMSE log (moyenne des plis)")
          .reset_index(drop=True))
combos.insert(0, "Rang", combos.index + 1)

# =======================================================================
# Rôles des combinaisons de référence
# =======================================================================
CHOICE = "Lasso + ElasticNet + Kernel Ridge + Gradient Boosting + XGBoost"
ROLES = {
    CHOICE: "Les cinq modèles (retenu)",
    "Lasso + Kernel Ridge + Gradient Boosting + XGBoost": "Les cinq sans ElasticNet",
    "Lasso + Kernel Ridge + XGBoost": "Les trois de l'expérience 06",
    "Lasso + Kernel Ridge + Gradient Boosting": "Variante avec Gradient Boosting",
    "Kernel Ridge": "Meilleur modèle seul",
}
combos.insert(1, "Rôle", combos["Combinaison"].map(ROLES).fillna(""))

def styled(table):
    # Tableau HTML sans index : ligne du modèle retenu en gras, nombres à 4 décimales.
    # Construit à la main (sans pandas Styler, qui exige jinja2); le texte n'est jamais coupé.
    def cell(value):
        return f"{value:.4f}" if isinstance(value, float) else escape(str(value))
    header = "".join(f"<th>{escape(str(column))}</th>" for column in table.columns)
    body = []
    for _, row in table.iterrows():
        style = ' style="font-weight: bold;"' if row["Combinaison"] == CHOICE else ""
        body.append(f"<tr{style}>" + "".join(f"<td>{cell(value)}</td>" for value in row) + "</tr>")
    return HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{''.join(body)}</tbody></table>")

SCORES = ["RMSE log (moyenne des plis)", "Écart type entre plis"]

show_title("4.1 Les 10 meilleures combinaisons sur 31")
display(styled(combos[["Rang", "Rôle", "Combinaison", "Nombre de modèles", "Familles", *SCORES]]
               .head(10)))

show_title("4.2 Combinaisons de référence, pli par pli")
display(styled(combos.loc[combos["Combinaison"].isin(ROLES),
                          ["Rang", "Combinaison", *FOLDS, *SCORES]]))

# =======================================================================
# Chaque modèle seul, pli par pli (combinaisons d'un seul modèle, déjà calculées)
# =======================================================================
singles = (combos[combos["Nombre de modèles"] == 1]
           .set_index("Combinaison").loc[list(MEMBERS)].reset_index())

show_title("4.3 Chaque modèle seul, pli par pli")
display(styled(singles[["Combinaison", *FOLDS, *SCORES]]))

1460 ventes étiquetées dans train.csv, 1459 maisons dans test.csv.


**4.1 Les 10 meilleures combinaisons sur 31**

Rang,Rôle,Combinaison,Nombre de modèles,Familles,RMSE log (moyenne des plis),Écart type entre plis
1,Les cinq modèles (retenu),Lasso + ElasticNet + Kernel Ridge + Gradient Boosting + XGBoost,5,Les deux,0.1074,0.0081
2,Les cinq sans ElasticNet,Lasso + Kernel Ridge + Gradient Boosting + XGBoost,4,Les deux,0.1074,0.0083
3,,ElasticNet + Kernel Ridge + Gradient Boosting + XGBoost,4,Les deux,0.1075,0.0084
4,Variante avec Gradient Boosting,Lasso + Kernel Ridge + Gradient Boosting,3,Les deux,0.1076,0.0080
5,,ElasticNet + Kernel Ridge + Gradient Boosting,3,Les deux,0.1076,0.0081
6,,Lasso + ElasticNet + Gradient Boosting + XGBoost,4,Les deux,0.1077,0.0084
7,Les trois de l'expérience 06,Lasso + Kernel Ridge + XGBoost,3,Les deux,0.1077,0.0076
8,,ElasticNet + Kernel Ridge + XGBoost,3,Les deux,0.1078,0.0076
9,,Lasso + ElasticNet + Kernel Ridge + Gradient Boosting,4,Les deux,0.1079,0.0078
10,,Lasso + ElasticNet + Kernel Ridge + XGBoost,4,Les deux,0.1081,0.0075


**4.2 Combinaisons de référence, pli par pli**

Rang,Combinaison,Pli 1,Pli 2,Pli 3,Pli 4,Pli 5,RMSE log (moyenne des plis),Écart type entre plis
1,Lasso + ElasticNet + Kernel Ridge + Gradient Boosting + XGBoost,0.1230,0.1023,0.1051,0.1066,0.1001,0.1074,0.0081
2,Lasso + Kernel Ridge + Gradient Boosting + XGBoost,0.1234,0.1021,0.1050,0.1071,0.0997,0.1074,0.0083
4,Lasso + Kernel Ridge + Gradient Boosting,0.1232,0.1029,0.1042,0.1069,0.1009,0.1076,0.0080
7,Lasso + Kernel Ridge + XGBoost,0.1224,0.1037,0.1058,0.1064,0.1004,0.1077,0.0076
25,Kernel Ridge,0.1230,0.1089,0.1058,0.1081,0.1044,0.1100,0.0067


**4.3 Chaque modèle seul, pli par pli**

Combinaison,Pli 1,Pli 2,Pli 3,Pli 4,Pli 5,RMSE log (moyenne des plis),Écart type entre plis
Lasso,0.1248,0.1075,0.1089,0.1089,0.1056,0.1111,0.0069
ElasticNet,0.1250,0.1075,0.1089,0.1089,0.1056,0.1112,0.0070
Kernel Ridge,0.1230,0.1089,0.1058,0.1081,0.1044,0.1100,0.0067
Gradient Boosting,0.1327,0.1063,0.1097,0.1163,0.1042,0.1138,0.0103
XGBoost,0.1289,0.1061,0.1137,0.1151,0.1020,0.1132,0.0092


**Lecture des tableaux.** Les 10 meilleures combinaisons mélangent toutes des modèles linéaires et des modèles d'arbres, et toutes font mieux que le meilleur modèle seul, Kernel Ridge (0,1100, 25ᵉ rang). C'est ce qu'annonçait la section 2 : la moyenne gagne surtout quand les modèles se trompent sur des maisons différentes. Les écarts entre les meilleures combinaisons sont toutefois très petits : de 0,1074 à 0,1081, bien moins que la variation d'un pli à l'autre (environ 0,008).

**Décision : nous retenons les cinq modèles**, pour trois raisons.

**1. C'est la meilleure combinaison.** Les cinq modèles arrivent au premier rang sur 31, avec une RMSE log de 0,1074. Ce résultat confirme celui de l'expérience 04, obtenu avec un autre prétraitement : la moyenne des cinq y faisait mieux que la moyenne des trois dans chacun des cinq plis (section 1.5).

**2. Les deux modèles d'arbres sont utiles.** Les trois modèles de l'expérience 06 (Lasso, Kernel Ridge et XGBoost) arrivent au 7ᵉ rang avec 0,1077, et la même combinaison avec Gradient Boosting au 4ᵉ rang avec 0,1076. Retirer Gradient Boosting des cinq modèles fait passer le score de 0,1074 à 0,1081 (10ᵉ rang). Avoir deux modèles d'arbres réduit un peu plus l'erreur, en particulier dans le pli 2 (0,1023 pour les cinq, contre 0,1037 pour les trois de l'expérience 06) : comme les deux modèles d'arbres ne se trompent pas tout à fait sur les mêmes maisons, leur moyenne compense une partie de leurs erreurs (section 2.1).

**3. ElasticNet ne coûte rien.** Sans lui, les quatre autres modèles obtiennent le même score (0,1074, 2ᵉ rang). ElasticNet n'apporte presque rien de nouveau, puisqu'il se trompe sur les mêmes maisons que Lasso; il revient surtout à donner un poids double à l'approche de Lasso. Nous le gardons, car le retirer ne change pas le score, et la moyenne des cinq reste le modèle évalué dans toutes nos expériences.

## 5. Modèle retenu

Le modèle retenu à la section 4 est la moyenne simple de cinq modèles : Lasso, ElasticNet, Kernel Ridge, Gradient Boosting et XGBoost. Chaque modèle prédit le logarithme du prix de la maison; nous faisons la moyenne de ces cinq prédictions, puis nous revenons en dollars.

Avant d'arriver aux modèles, chaque maison passe par le prétraitement de l'étape 4, adapté à la famille du modèle. Les trois modèles linéaires le reçoivent au complet, avec le logarithme des colonnes à longue queue et la standardisation. Les deux modèles d'arbres le reçoivent sans ces deux transformations, qui ne changent rien pour un arbre. Les deux ventes aberrantes (524 et 1299) sont retirées de l'entraînement seulement.

Cette section présente ce modèle en trois parties, qui répondent à la consigne : son schéma (5.1), les paramètres utilisés et la façon dont ils ont été choisis (5.2), puis les hypothèses retenues (5.3).

### 5.1 Schéma

```text
Maison brute (79 colonnes)
   │
   ├── make_base() complet : nettoyage, valeurs manquantes, variables créées,
   │   encodage, log1p des longues queues, standardisation
   │      ├── Lasso ─────────────┐
   │      ├── ElasticNet ────────┤
   │      └── Kernel Ridge ──────┤
   │                             ├──► moyenne des 5 log(prix) ──► exp ──► prix prédit
   └── make_base() sans log1p     │
       ni standardisation         │
          ├── Gradient Boosting ─┤
          └── XGBoost ───────────┘
```

- **Prétraitement.** Chaque modèle a son propre pipeline (`make_preprocessing` dans `src/training.py`), appris sur les maisons d'entraînement seulement.
- **Cible.** Chaque modèle apprend `log(SalePrice)`. `TransformedTargetRegressor` applique le logarithme à l'entraînement et l'exponentielle à la prédiction.
- **Combinaison.** `VotingRegressor` fait la moyenne des cinq prédictions sur l'échelle logarithmique, celle de la métrique. Sur l'échelle des prix, c'est une moyenne géométrique.
- **Ventes aberrantes.** Les maisons 524 et 1299 (plus de 4 000 pi², vendues moins de 300 000 \$) sont retirées de l'entraînement (`fit_model`), jamais des maisons évaluées ni de `test.csv`.

### 5.2 Paramètres utilisés

| Modèle | Paramètres | Comment ils ont été choisis |
|---|---|---|
| Lasso | Pénalité $\alpha$ parmi 40 valeurs de $10^{-4,5}$ à $10^{-1,5}$ | Déterminée pendant l'entraînement, par validation croisée sur une grille |
| ElasticNet | Même grille de pénalités; part de Lasso parmi 0,1, 0,3, 0,5, 0,7, 0,9 et 1,0 | Déterminées pendant l'entraînement, par validation croisée sur une grille |
| Kernel Ridge | Noyau polynomial de degré 2, $\alpha$ = 0,6, `coef0` = 2,5 | Valeurs usuelles pour ce jeu de données |
| Gradient Boosting | 3 000 arbres de profondeur 4, taux d'apprentissage 0,05, perte de Huber, au moins 15 ventes par feuille | Valeurs usuelles pour ce jeu de données |
| XGBoost | 2 000 arbres de profondeur 4, taux d'apprentissage 0,02, 70 % des ventes et 50 % des colonnes par arbre | Configuration manuelle de l'expérience 03; une recherche aléatoire n'y gagnait que 0,0006 |

Pour Lasso et ElasticNet, nous ne fixons pas la pénalité : nous donnons une grille de valeurs candidates, et le modèle garde, pendant l'entraînement, celle qui donne la plus petite erreur en validation croisée.

[Les paramètres sont définis dans `src/training.py`](../src/training.py), dans le dictionnaire `MEMBERS`. Par exemple :

```python
PENALTIES = np.logspace(-4.5, -1.5, 40)   # grille de 40 pénalités candidates

MEMBERS = {
    "Lasso": LassoCV(alphas=PENALTIES, max_iter=50_000),
    "ElasticNet": ElasticNetCV(l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 1.0], alphas=PENALTIES,
                               max_iter=50_000),
    "Kernel Ridge": KernelRidge(alpha=0.6, kernel="polynomial", degree=2, coef0=2.5),
    ...
}
```

**Graine.** `RANDOM_STATE = 42` rend les résultats reproductibles. **Combinaison.** Moyenne simple : chaque modèle compte pour 1/5.

La cellule suivante montre les valeurs retenues par Lasso et ElasticNet après l'entraînement.

In [5]:
show_title("5.2 Pénalités choisies par validation croisée interne")
linear = fit_model(make_model(["Lasso", "ElasticNet"]), X, y)
fitted = linear.regressor_.named_estimators_   # un pipeline par modèle : prétraitement, puis modèle
lasso, elastic_net = fitted["Lasso"][-1], fitted["ElasticNet"][-1]

display(pd.DataFrame({
    "Pénalité alpha choisie": [f"{lasso.alpha_:.5f}", f"{elastic_net.alpha_:.5f}"],
    "Part de Lasso (l1_ratio) choisie": ["1 (Lasso seul)", f"{elastic_net.l1_ratio_:g}"],
    "Coefficients non nuls": [f"{(lasso.coef_ != 0).sum()} sur {lasso.coef_.size}",
                              f"{(elastic_net.coef_ != 0).sum()} sur {elastic_net.coef_.size}"],
}, index=["Lasso", "ElasticNet"]))

**5.2 Pénalités choisies par validation croisée interne**

,Pénalité alpha choisie,Part de Lasso (l1_ratio) choisie,Coefficients non nuls
Lasso,0.00064,1 (Lasso seul),93 sur 227
ElasticNet,0.00064,1,93 sur 227


**Lecture.** Lasso retient une pénalité de 0,00064 et garde 93 des 227 colonnes. ElasticNet retient la même pénalité et une part de Lasso de 1 : sur nos données, il devient exactement Lasso, d'où la corrélation de 1,00 entre leurs erreurs (section 1.4).

### 5.3 Hypothèses retenues

**Hypothèses sur les données.**

1. **Échelle logarithmique.** Le prix est asymétrique (étape 3), et la métrique de Kaggle mesure des écarts relatifs. Modéliser `log(SalePrice)` rend les erreurs comparables entre maisons bon marché et chères.
2. **Ventes typiques.** Les maisons 524 et 1299 sont considérées hors de la population à prédire, comme le recommande l'auteur du jeu de données. Elles sont retirées de l'entraînement, et les scores de validation croisée sont calculés sans elles (`cv_rmse`, étape 4).
3. **Stabilité du marché.** Les ventes de 2006 à 2010 à Ames, y compris pendant la crise de 2008, représentent les ventes de `test.csv`, qui couvrent la même période.
4. **Erreurs complémentaires.** Les modèles linéaires et les arbres ne se trompent pas sur les mêmes maisons. En faire la moyenne compense une partie de leurs erreurs (vérifié aux sections 1.4 et 4).

**Hypothèses de chaque modèle.**

| Modèle | Hypothèse |
|---|---|
| Lasso | Le logarithme du prix est une somme d'effets, et une partie seulement des colonnes est utile. |
| ElasticNet | Même hypothèse, avec des groupes de colonnes liées qui se partagent l'effet. |
| Kernel Ridge | Le logarithme du prix est une fonction régulière des caractéristiques, et les interactions entre deux caractéristiques suffisent à décrire leurs effets combinés. |
| Gradient Boosting et XGBoost | Le prix change par seuils, et les maisons à prédire restent dans l'étendue des ventes d'entraînement, puisque les arbres ne prolongent pas les tendances. |

## 6. Résultats et entraînement final

Le modèle est entraîné une seule fois sur les 1 460 ventes de `train.csv`, sans les deux ventes aberrantes, pour prédire les 1 459 maisons de `test.csv`. Le fichier suit le format de `sample_submission.csv`. L'évaluation sur Kaggle relève de l'[étape 6](06_evaluation.ipynb).

In [6]:
show_title("6.1 Soumission Kaggle")
final_model = fit_model(make_model(list(MEMBERS)), X, y)
submission = pd.DataFrame({"Id": test["Id"], "SalePrice": final_model.predict(test)})

sample = pd.read_csv(root / "data/sample_submission.csv")
assert list(submission.columns) == list(sample.columns)
assert submission["Id"].equals(sample["Id"])
assert submission["SalePrice"].gt(0).all()

submission_path = root / "submissions" / "05_ensemble.csv"
submission_path.parent.mkdir(exist_ok=True)
submission.to_csv(submission_path, index=False)
median = f"{submission['SalePrice'].median():,.0f}".replace(",", " ")
print(f"{submission_path.relative_to(root)} : {len(submission)} prix, médiane {median} $")

**6.1 Soumission Kaggle**

submissions\05_ensemble.csv : 1459 prix, médiane 157 893 $


**Bilan.**

- **Démarche.** Le RAG, efficace sur des annonces décrites par du texte, convient mal à des données purement tabulaires (0,140, puis 0,133 avec de meilleurs comparables). XGBoost, conçu pour ce type de données, fait mieux (0,125 à 0,130). Un ensemble de modèles de deux familles fait mieux encore : 0,1233 en validation pour cinq modèles, 0,1222 pour trois.
- **Modèle retenu.** `make_model()` dans `src/training.py` : la moyenne des log(prix) prédits par Lasso, ElasticNet, Kernel Ridge, Gradient Boosting et XGBoost, chacun avec le prétraitement de l'étape 4 adapté à sa famille. En validation croisée sur les 1 460 ventes, il obtient une RMSE log de 0,1074, la meilleure des 31 combinaisons possibles des cinq modèles.
- **Limites.** Kernel Ridge et Gradient Boosting gardent des paramètres usuels, sans recherche. Lasso et ElasticNet sont presque le même modèle : la moyenne donne en pratique un poids double à cette approche. Les écarts entre combinaisons (quelques dix-millièmes) sont plus petits que l'écart type entre plis. Enfin, les scores de validation croisée ont servi à choisir les modèles : ils sont légèrement optimistes, et seule la soumission Kaggle (étape 6) donne une mesure indépendante.

## Références

- Chen, T. et Guestrin, C. (2016). « XGBoost: A Scalable Tree Boosting System ». *Proceedings of the 22nd ACM SIGKDD International Conference on Knowledge Discovery and Data Mining*, p. 785–794.
- De Cock, D. (2011). « Ames, Iowa: Alternative to the Boston Housing Data as an End of Semester Regression Project ». *Journal of Statistics Education*, 19(3). https://doi.org/10.1080/10691898.2011.11889627
- Friedman, J. H. (2001). « Greedy Function Approximation: A Gradient Boosting Machine ». *The Annals of Statistics*, 29(5), p. 1189–1232.
- Grinsztajn, L., Oyallon, E. et Varoquaux, G. (2022). « Why Do Tree-Based Models Still Outperform Deep Learning on Typical Tabular Data? ». *Advances in Neural Information Processing Systems 35 (NeurIPS 2022)*.
- Huber, P. J. (1964). « Robust Estimation of a Location Parameter ». *The Annals of Mathematical Statistics*, 35(1), p. 73–101.
- Krogh, A. et Vedelsby, J. (1995). « Neural Network Ensembles, Cross Validation, and Active Learning ». *Advances in Neural Information Processing Systems 7 (NIPS 1994)*, p. 231–238.
- Saunders, C., Gammerman, A. et Vovk, V. (1998). « Ridge Regression Learning Algorithm in Dual Variables ». *Proceedings of the 15th International Conference on Machine Learning (ICML '98)*, p. 515–521.
- Tibshirani, R. (1996). « Regression Shrinkage and Selection via the Lasso ». *Journal of the Royal Statistical Society, Series B*, 58(1), p. 267–288.
- Zou, H. et Hastie, T. (2005). « Regularization and Variable Selection via the Elastic Net ». *Journal of the Royal Statistical Society, Series B*, 67(2), p. 301–320.
- Expériences du projet : `notebooks/experiences/01_rag_filtre.ipynb` à `06_ensemble_trois_modeles.ipynb`.